In [ ]:
import pandas as pd

file_path = '/content/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx'

# Load the Excel file to inspect sheets
excel_file = pd.ExcelFile(file_path)
sheet_names = excel_file.sheet_names

print(f"Sheets in the Excel file: {sheet_names}")

# Load the first sheet into a DataFrame
df = pd.read_excel(file_path, sheet_name=sheet_names[0])

# Display the first 5 rows of the DataFrame
display(df.head())

Sheets in the Excel file: ['products', 'customers', 'orders', 'transactions', 'returns', 'vendors', 'customer_reviews', 'order_payments']


,product_id,product_name,colors,category,sub_category,date_added,manufacturer,sizes,upc,weight,product_photos_qty,Unnamed: 11
0,FUR-BO-10000330,"Sauder Camden County Barrister Bookcase, Plank...",Blue,Furniture,Bookcases,2016-03-30,NaN,NaN,NaN,NaN,3,NaN
1,FUR-BO-10000362,Sauder Inglewood Library Bookcases,Blue,Furniture,Bookcases,2015-08-17,2(x)ist,NaN,NaN,NaN,6,NaN
2,FUR-BO-10000468,O'Sullivan 2-Shelf Heavy-Duty Bookcases,Black,Furniture,Bookcases,2016-01-15,NaN,22,NaN,NaN,4,NaN
3,FUR-BO-10000711,"Hon Metal Bookcases, Gray",Blue,Furniture,Bookcases,2016-08-23,NaN,NaN,NaN,NaN,2,NaN
4,FUR-BO-10000780,O'Sullivan Plantations 2-Door Library in Landv...,Pink,Furniture,Bookcases,2016-01-14,NaN,NaN,NaN,NaN,0,NaN


### Step 1: Analyze Customer History

In [ ]:
# Load all necessary sheets into separate DataFrames
customers_df = pd.read_excel(file_path, sheet_name='customers')
orders_df = pd.read_excel(file_path, sheet_name='orders')
products_df = pd.read_excel(file_path, sheet_name='products')
transactions_df = pd.read_excel(file_path, sheet_name='transactions')
returns_df = pd.read_excel(file_path, sheet_name='returns')
vendors_df = pd.read_excel(file_path, sheet_name='vendors')
customer_reviews_df = pd.read_excel(file_path, sheet_name='customer_reviews')
order_payments_df = pd.read_excel(file_path, sheet_name='order_payments')

print("DataFrames loaded:")
for name, df in {'customers': customers_df, 'orders': orders_df, 'products': products_df, 'transactions': transactions_df, 'returns': returns_df, 'vendors': vendors_df, 'customer_reviews': customer_reviews_df, 'order_payments': order_payments_df}.items():
    print(f"  {name}: {df.shape[0]} rows, {df.shape[1]} columns")


DataFrames loaded:
  customers: 793 rows, 9 columns
  orders: 5015 rows, 10 columns
  products: 1774 rows, 12 columns
  transactions: 9819 rows, 7 columns
  returns: 296 rows, 2 columns
  vendors: 5 rows, 2 columns
  customer_reviews: 1597 rows, 5 columns
  order_payments: 4914 rows, 6 columns


In [ ]:
# Convert 'order_purchase_date' to datetime objects
orders_df['order_purchase_date'] = pd.to_datetime(orders_df['order_purchase_date'])

# Filter orders for the last six months
last_six_months = orders_df['order_purchase_date'].max() - pd.DateOffset(months=6)
recent_orders_df = orders_df[orders_df['order_purchase_date'] >= last_six_months]

# Calculate total sales amount per order from transactions_df
order_sales_amount = transactions_df.groupby('order_id')['sales_amt'].sum().reset_index()
order_sales_amount = order_sales_amount.rename(columns={'sales_amt': 'order_total_amount'})

# Merge recent_orders_df with order_sales_amount to get the total amount for each recent order
recent_orders_with_amount_df = pd.merge(recent_orders_df, order_sales_amount, on='order_id', how='left')

# Calculate total_spending and order_count for each customer
customer_history = recent_orders_with_amount_df.groupby('customer_id').agg(
    total_spending=('order_total_amount', 'sum'),
    order_count=('order_id', 'count')
).reset_index()

display(customer_history.head())

,customer_id,total_spending,order_count
0,AA-10315,2637.518,1
1,AA-10375,29.320,3
2,AA-10480,2235.244,2
3,AA-10645,230.895,2
4,AB-10015,77.144,2


### Step 2: Assign Loyalty Tiers And Discounts

In [ ]:
# Merge customer_history with customers_df to get customer names
customer_loyalty = pd.merge(customer_history, customers_df[['customer_id', 'customer_name']], on='customer_id', how='left')

# Define a function to assign loyalty tier and discount
def assign_loyalty(row):
    spending = row['total_spending']
    orders = row['order_count']

    if spending < 500:
        tier = 'Silver'
        discount = 2 if orders < 10 else 4
    elif 500 <= spending <= 2000:
        tier = 'Gold'
        discount = 6 if orders < 10 else 8
    else: # spending > 2000
        tier = 'Platinum'
        discount = 10 if orders < 10 else 15
    return tier, discount

# Apply the function to create 'tier' and 'discount_applicable' columns
customer_loyalty[['tier', 'discount_applicable']] = customer_loyalty.apply(lambda row: pd.Series(assign_loyalty(row)), axis=1)

display(customer_loyalty.head())

,customer_id,total_spending,order_count,customer_name,tier,discount_applicable
0,AA-10315,2637.518,1,Alex Avila,Platinum,10
1,AA-10375,29.320,3,Allen Armold,Silver,2
2,AA-10480,2235.244,2,Andrew Allen,Platinum,10
3,AA-10645,230.895,2,Anna Andreadi,Silver,2
4,AB-10015,77.144,2,Aaron Bergman,Silver,2


### Step 3: Build The Customer Loyalty Tier Report

In [ ]:
# The 'customer_loyalty' DataFrame already contains the required information for the report
display(customer_loyalty[['customer_id', 'customer_name', 'total_spending', 'order_count', 'tier', 'discount_applicable']].head())

,customer_id,customer_name,total_spending,order_count,tier,discount_applicable
0,AA-10315,Alex Avila,2637.518,1,Platinum,10
1,AA-10375,Allen Armold,29.320,3,Silver,2
2,AA-10480,Andrew Allen,2235.244,2,Platinum,10
3,AA-10645,Anna Andreadi,230.895,2,Silver,2
4,AB-10015,Aaron Bergman,77.144,2,Silver,2


### Step 4: Apply Discounts To New Orders

In [ ]:
# Load the new orders dataset
new_orders_df = pd.read_csv('/content/517cad90-cfb3-48fe-975e-256d942412ac_neworders.csv')

# Merge the loyalty report with the new orders dataset
# We need customer_name, so let's merge customer_loyalty with new_orders_df
discounted_orders_df = pd.merge(
    new_orders_df,
    customer_loyalty[['customer_id', 'customer_name', 'discount_applicable']],
    on='customer_id',
    how='left'
)

# Calculate the discounted price
discounted_orders_df['discounted_price'] = discounted_orders_df['original_price'] * (1 - discounted_orders_df['discount_applicable'] / 100)

display(discounted_orders_df.head())

,customer_id,Order_id,original_price,customer_name,discount_applicable,discounted_price
0,AB-10105,CA-2014-103392,55.462,Adrian Barton,6.0,52.13428
1,DM-13525,CA-2014-103254,66.374,Don Miller,6.0,62.39156
2,RT-13456,CA-2014-103445,588.678,NaN,NaN,NaN
3,JH-10250,CA-2014-103136,224.731,NaN,NaN,NaN
4,AB-10105,CA-2014-103283,578.572,Adrian Barton,6.0,543.85768


### Step 5: Build The Discounted Pricing Report

In [ ]:
discounted_pricing_report = discounted_orders_df[[
    'Order_id',
    'customer_id',
    'customer_name',
    'original_price',
    'discount_applicable',
    'discounted_price'
]]

display(discounted_pricing_report.head())

,Order_id,customer_id,customer_name,original_price,discount_applicable,discounted_price
0,CA-2014-103392,AB-10105,Adrian Barton,55.462,6.0,52.13428
1,CA-2014-103254,DM-13525,Don Miller,66.374,6.0,62.39156
2,CA-2014-103445,RT-13456,NaN,588.678,NaN,NaN
3,CA-2014-103136,JH-10250,NaN,224.731,NaN,NaN
4,CA-2014-103283,AB-10105,Adrian Barton,578.572,6.0,543.85768


In [13]:
print(f"Shape of recent_orders_df (orders in last six months): {recent_orders_df.shape}")
distinct_orders_count = recent_orders_df['order_id'].nunique()
print(f"Distinct orders remaining after filtering to the last six months: {distinct_orders_count}")

Shape of recent_orders_df (orders in last six months): (2041, 10)
Distinct orders remaining after filtering to the last six months: 2039


In [14]:
print(orders_df.columns)

Index(['order_id', 'customer_id', 'ship_mode', 'vendor_id', 'order_status',
       'order_purchase_date', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date'],
      dtype='object')
